# B05 · S2 — Sistemas expertos (Pagarium)

**Objetivo (RA5-a/c):** entender la anatomía de un sistema experto y el ciclo *reconocer-resolver-actuar* construyendo un micro-motor, viendo **por qué falla** y comparándolo con `experta` y CLIPS.

> Hilo conductor: **Pagarium**, una pasarela de pagos. Práctica guiada de la S2 de los [apuntes](../apuntes.md).

## 1. Un micro-motor en 35 líneas (falla a propósito)

Dispara **todas** las reglas que encajan, sin resolver conflictos: dos decisiones contradictorias sobre el mismo pago.

In [ ]:
class MicroMotor:
    def __init__(self):
        self.hechos, self.reglas, self.traza = {}, [], []

    def hecho(self, **h):
        self.hechos.update(h)

    def regla(self, nombre, prioridad, condicion, accion):
        self.reglas.append((nombre, prioridad, condicion, accion))

    def run(self):
        pendientes = True
        while pendientes:
            pendientes = False
            for nombre, prioridad, condicion, accion in self.reglas:
                if nombre in self.traza:
                    continue
                if condicion(self.hechos):
                    self.traza.append(nombre)
                    accion(self.hechos)
                    pendientes = True
        return self.hechos

motor = MicroMotor()
motor.hecho(importe=200, antiguedad_meses=10, n_intentos=5)
motor.regla("aprobar_pequena", 10, lambda h: h["importe"] < 500,
            lambda h: h.update(decision="aprobar"))
motor.regla("rechazar_reintentos", 10, lambda h: h["n_intentos"] >= 4,
            lambda h: h.update(decision="rechazar"))
print(motor.run())
print(motor.traza)

## 2. Actividad — corregir el conflicto con `salience`

Modifica `run()` para que, cuando varias reglas encajen, se dispare **solo la de mayor prioridad**. Comprueba que ya no hay contradicción.

In [ ]:
# TODO: reescribe run() con resolución de conflictos por prioridad
class MicroMotor2:
    ...

## 3. La misma política con `experta`

In [ ]:
%pip install experta

import collections, collections.abc
if not hasattr(collections, "Mapping"):
    collections.Mapping = collections.abc.Mapping
    collections.Iterable = collections.abc.Iterable
    collections.MutableMapping = collections.abc.MutableMapping

from experta import *

class Pagarium(KnowledgeEngine):
    @DefFacts()
    def inicio(self):
        yield Fact(accion="evaluar")

    @Rule(Fact(accion="evaluar"), Fact(importe=P(lambda i: i < 500)),
          Fact(n_intentos=P(lambda n: n < 4)), salience=20)
    def aprobar(self):
        self.declare(Fact(decision="aprobar"))

    @Rule(Fact(n_intentos=P(lambda n: n >= 4)), salience=30)
    def rechazar(self):
        self.declare(Fact(decision="rechazar"))

motor = Pagarium(); motor.reset()
motor.declare(Fact(importe=200, n_intentos=5))
motor.run()
print([dict(f) for f in motor.facts.values() if "decision" in f])

## 4. La misma política en CLIPS (`clipspy`)

In [ ]:
%pip install clipspy
import clips

env = clips.Environment()
env.build("""
(defrule aprobar
  (importe ?i&:(< ?i 500))
  (n_intentos ?n&:(< ?n 4))
  =>
  (assert (decision aprobar)))
(defrule rechazar
  (declare (salience 30))
  (n_intentos ?n&:(>= ?n 4))
  =>
  (assert (decision rechazar)))
""")
env.assert_string("(importe 200)")
env.assert_string("(n_intentos 5)")
env.run()
print([str(f) for f in env.facts()])

**Cierre:** compara la traza de los tres motores y explica en 3 líneas qué papel juega la **resolución de conflictos**.